# Demag re-run — detailed analysis

기본 스키마와 파싱 / 4 calf τ 시계열은 [`view_data.ipynb`](./view_data.ipynb) 에 있음. 이 노트북은 그 위에 얹는 **상세 분석** 만 담음.

- §1. 준비 & 사용 가능한 파일 일람
- §2. 파일 한 개 — 메타 / 키 통계
- §3. 12 관절 전체 q / qd
- §4. 베이스 상태 (pos / quat / rpy / lin·ang vel / xy trajectory)
- §5. 12 감자 케이스 ratio 요약표
- §6. 4 × 3 summary grid
- §7. PD baseline vs MethodA healthy

## 1. 준비 & 파일 일람

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path("./data")
LEGS = ("FL", "FR", "RL", "RR")
FACTORS = (1.0, 0.8, 0.6, 0.4)

pd.set_option("display.max_rows", 60)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

def load_npz(relpath: str) -> dict:
    raw = np.load(DATA_DIR / relpath, allow_pickle=True)
    out = {k: raw[k] for k in raw.files if k != "meta"}
    out["meta"] = json.loads(str(raw["meta"]))
    out["time"] = np.arange(out["meta"]["num_steps"]) * out["meta"]["dt"]
    return out

def col_of(d: dict, joint_suffix: str) -> int:
    names = d["meta"]["q_all_joint_names"]
    for i, n in enumerate(names):
        if n in (joint_suffix, f"{joint_suffix}_joint"):
            return i
    raise KeyError(f"{joint_suffix} not in {names}")

def calf_col(d, leg):  return col_of(d, f"{leg}_calf")
def hip_col(d, leg):   return col_of(d, f"{leg}_hip")
def thigh_col(d, leg): return col_of(d, f"{leg}_thigh")

rows = []
for p in sorted(DATA_DIR.glob("**/*.npz")):
    rows.append({"file": str(p.relative_to(DATA_DIR)),
                 "size_KB": round(p.stat().st_size / 1024, 1)})
pd.DataFrame(rows)

## 2. 파일 한 개 — 메타 / 키 통계

`FILE_TO_VIEW` 값을 바꾸면 다른 조건 파일도 볼 수 있음.

In [ ]:
FILE_TO_VIEW = "methoda/FL_0.6.npz"

data = load_npz(FILE_TO_VIEW)
print(f"=== meta ({FILE_TO_VIEW}) ===")
for k, v in data["meta"].items():
    print(f"  {k}: {v}")

In [ ]:
rows = []
for k in sorted(data):
    if k in ("meta", "time"):
        continue
    a = np.asarray(data[k])
    rows.append({"key": k, "shape": str(a.shape), "dtype": str(a.dtype),
                 "min": float(a.min()), "max": float(a.max()), "mean": float(a.mean())})
pd.DataFrame(rows)

## 3. 12 관절 전체 q / qd

감자 calf 만 굵게 그려 대비 강조.

In [ ]:
def plot_q_qd_all(d, title):
    names = d["meta"]["q_all_joint_names"]
    demag_leg = d["meta"].get("demag_leg", "none")
    demag_calf = f"{demag_leg}_calf_joint"
    fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
    for c in range(12):
        is_demag = names[c] == demag_calf
        kw = {"lw": 1.8, "alpha": 0.95} if is_demag else {"lw": 0.8, "alpha": 0.6}
        axes[0].plot(d["time"], d["q_all"][:, c], label=names[c], **kw)
        axes[1].plot(d["time"], d["qd_all"][:, c], label=names[c], **kw)
    axes[0].set_ylabel("q [rad]")
    axes[1].set_ylabel("qd [rad/s]")
    axes[1].set_xlabel("time [s]")
    axes[0].legend(ncol=4, fontsize=7, loc="upper right")
    for ax in axes: ax.grid(alpha=0.3)
    fig.suptitle(title, fontsize=11)
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

plot_q_qd_all(data, f"{FILE_TO_VIEW} — all 12 joints")

## 4. 베이스 상태

위치(world) / 쿼터니언 / RPY / 속도(body) / xy 궤적.

In [ ]:
def plot_base_state(d, title):
    fig, axes = plt.subplots(2, 3, figsize=(16, 6), sharex=True)
    for i, lbl in enumerate(["x (world)", "y (world)", "z (world)"]):
        axes[0, 0].plot(d["time"], d["base_pos"][:, i], lw=1.2, label=lbl)
    axes[0, 0].set_title("base_pos  [m]"); axes[0, 0].legend(fontsize=8)
    for i, lbl in enumerate(["qw", "qx", "qy", "qz"]):
        axes[0, 1].plot(d["time"], d["base_quat"][:, i], lw=1.1, label=lbl)
    axes[0, 1].set_title("base_quat  [wxyz]"); axes[0, 1].legend(fontsize=8, ncol=2)
    rpy_deg = np.rad2deg(d["base_rpy"])
    for i, lbl in enumerate(["roll", "pitch", "yaw"]):
        axes[0, 2].plot(d["time"], rpy_deg[:, i], lw=1.2, label=lbl)
    axes[0, 2].set_title("base_rpy  [deg, ZYX intrinsic]"); axes[0, 2].legend(fontsize=8)
    for i, lbl in enumerate(["vx (body)", "vy (body)", "vz (body)"]):
        axes[1, 0].plot(d["time"], d["base_lin_vel"][:, i], lw=1.2, label=lbl)
    axes[1, 0].plot(d["time"], d["cmd_vel"][:, 0], color="k", lw=0.8, ls=":", label="cmd vx")
    axes[1, 0].set_title("base_lin_vel  [m/s]"); axes[1, 0].legend(fontsize=8)
    for i, lbl in enumerate(["wx (body)", "wy (body)", "wz (body)"]):
        axes[1, 1].plot(d["time"], d["base_ang_vel"][:, i], lw=1.2, label=lbl)
    axes[1, 1].plot(d["time"], d["cmd_vel"][:, 2], color="k", lw=0.8, ls=":", label="cmd wz")
    axes[1, 1].set_title("base_ang_vel  [rad/s]"); axes[1, 1].legend(fontsize=8)
    axes[1, 2].plot(d["base_pos"][:, 0], d["base_pos"][:, 1], lw=1.5, color="tab:purple")
    axes[1, 2].set_title("base xy trajectory (world)"); axes[1, 2].set_aspect("equal")
    axes[1, 2].set_xlabel("x [m]"); axes[1, 2].set_ylabel("y [m]")
    for r in range(2):
        for c in range(3):
            axes[r, c].grid(alpha=0.3)
    axes[1, 0].set_xlabel("time [s]"); axes[1, 1].set_xlabel("time [s]")
    fig.suptitle(title, fontsize=11)
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

plot_base_state(data, f"{FILE_TO_VIEW} — base state")

## 5. 12 감자 케이스 ratio 요약표

정상 상태 구간(마지막 200 step) 에서 `mean|τ_actual| / mean|τ_cmd|` 을 측정 → 스펙 factor 와의 오차.

In [ ]:
rows = []
for leg in LEGS:
    for f in (0.8, 0.6, 0.4):
        p = DATA_DIR / "methoda" / f"{leg}_{f:.1f}.npz"
        if not p.exists():
            continue
        d = load_npz(p.relative_to(DATA_DIR).as_posix())
        c = calf_col(d, leg)
        tc = np.mean(np.abs(d["tau_cmd"][-200:, c]))
        ta = np.mean(np.abs(d["tau_actual"][-200:, c]))
        rows.append({"leg": leg, "factor": f,
                     "mean|τ_cmd|_end": tc, "mean|τ_actual|_end": ta,
                     "measured_ratio": ta / max(tc, 1e-6),
                     "abs_err_vs_spec": abs(ta / max(tc, 1e-6) - f)})
pd.DataFrame(rows)

## 6. 4 × 3 summary grid

행 = 다리, 열 = factor. 각 칸에 healthy τ_actual 을 회색 점선으로 깔고 감자 case 의 τ_cmd / τ_actual 을 겹쳐 그림.

In [ ]:
healthy = load_npz("methoda/healthy.npz")

fig, axes = plt.subplots(4, 3, figsize=(15, 10), sharex=True)
COLOR = {0.8: "#1f77b4", 0.6: "#ff7f0e", 0.4: "#d62728"}
for r, leg in enumerate(LEGS):
    for c, f in enumerate((0.8, 0.6, 0.4)):
        ax = axes[r, c]
        h_idx = calf_col(healthy, leg)
        ax.plot(healthy["time"], healthy["tau_actual"][:, h_idx],
                color="0.6", lw=1.0, ls=":", alpha=0.7, label="healthy τ_actual")
        p = DATA_DIR / "methoda" / f"{leg}_{f:.1f}.npz"
        if p.exists():
            d = load_npz(p.relative_to(DATA_DIR).as_posix())
            idx = calf_col(d, leg)
            ax.plot(d["time"], d["tau_cmd"][:, idx], color=COLOR[f], lw=1.0, ls="--", alpha=0.85, label="τ_cmd")
            ax.plot(d["time"], d["tau_actual"][:, idx], color=COLOR[f], lw=1.3, ls="-",  alpha=0.9,  label="τ_actual")
            ta = np.mean(np.abs(d["tau_actual"][-200:, idx]))
            tc = np.mean(np.abs(d["tau_cmd"][-200:, idx]))
            ratio = ta / max(tc, 1e-6)
            ax.set_title(f"{leg} ×{f:.1f}   ratio={ratio:.2f}", fontsize=10)
        else:
            ax.text(0.5, 0.5, "no data", transform=ax.transAxes, ha="center", va="center", color="0.6")
        ax.grid(alpha=0.3)
        if c == 0:
            ax.set_ylabel(f"{leg}\nτ [N·m]")
        if r == 3:
            ax.set_xlabel("time [s]")
axes[0, 0].legend(loc="upper right", fontsize=7)
fig.suptitle("MethodA demag summary grid", fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## 7. PD baseline vs MethodA healthy

In [ ]:
pd_base = load_npz("pd/nominal.npz")
ma_base = load_npz("methoda/healthy.npz")

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for d, col, label in ((pd_base, "tab:blue", "PD"), (ma_base, "tab:red", "MethodA")):
    calf_idx = [calf_col(d, l) for l in LEGS]
    mean_tau = np.mean(np.abs(d["tau_actual"][:, calf_idx]), axis=1)
    axes[0].plot(d["time"], mean_tau, color=col, lw=1.4, label=label)
    axes[1].plot(d["time"], d["base_pos"][:, 2], color=col, lw=1.4, label=label)
    axes[2].plot(d["time"], d["base_lin_vel"][:, 0], color=col, lw=1.4, label=label)
axes[2].plot(ma_base["time"], ma_base["cmd_vel"][:, 0], color="k", lw=0.8, ls=":", label="cmd")
axes[0].set_title("mean |τ_actual| on calves [N·m]")
axes[1].set_title("base z [m]")
axes[2].set_title("base vx [m/s]")
for ax in axes:
    ax.grid(alpha=0.3); ax.set_xlabel("time [s]"); ax.legend(fontsize=9)
fig.suptitle("PD baseline vs MethodA healthy", fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

## 응용 스니펫

- 특정 관절 뽑기:
  ```python
  d = load_npz("methoda/FL_0.4.npz")
  c = calf_col(d, "FL")          # 'FL_calf_joint' 열 인덱스
  tau_FL_calf = d["tau_actual"][:, c]
  ```
- 전체 관절 이름: `d["meta"]["q_all_joint_names"]`
- Saturation 비율 (Go2 calf 기준):
  ```python
  I_max = 45.0 / (0.128 * 6.33)
  sat = np.mean(np.abs(d["I_actual"]) >= 0.95 * I_max)
  ```